# sync_controls

> Synced navigation toggle for the combined Phase 2 dual-column step

In [ ]:
#| default_exp components.sync_controls

In [ ]:
#| export
from typing import Any

from fasthtml.common import Button, Script

from cjm_fasthtml_daisyui.components.actions.button import btn, btn_sizes, btn_styles, btn_colors
from cjm_fasthtml_daisyui.utilities.semantic_colors import bg_dui
from cjm_fasthtml_tailwind.utilities.spacing import m
from cjm_fasthtml_tailwind.core.base import combine_classes

from cjm_fasthtml_lucide_icons.factory import lucide_icon

# Design system recipes (V11 icon-size roles)
from cjm_fasthtml_design_system.icons import icons

from cjm_fasthtml_card_stack.js.sync import generate_card_stack_sync_js

## Constants

Shared between the sync JS generator, KeyAction, and toggle button.

In [ ]:
#| export
# JS function name for toggling sync (called by KeyAction wrapper)
SYNC_TOGGLE_FN = "toggleSegAlignSync"

# Window key for sync state + cleanup
SYNC_KEY = "_segAlignSync"

# HTML ID for the toolbar sync toggle button
SYNC_BTN_ID = "sd-sync-toggle-btn"

## Sync Toggle Button

Toolbar button that toggles synced navigation. Uses `btn-outline` when off,
`btn-primary` when on. The onclick handler calls the toggle function from
`generate_card_stack_sync_js()` and updates button styling.

In [ ]:
#| export
def render_sync_toggle_button() -> Any:  # Sync toggle button element
    """Render the synced navigation toggle button for the seg toolbar."""
    return Button(
        lucide_icon("link", size=icons.text_button, cls=str(m.r(1))),
        "Sync",
        id=SYNC_BTN_ID,
        type="button",
        cls=combine_classes(btn, btn_sizes.sm, btn_styles.outline),
        onclick=(
            f"var on=window.{SYNC_TOGGLE_FN}&&window.{SYNC_TOGGLE_FN}();"
            f"this.classList.toggle('btn-primary',on);"
            f"this.classList.toggle('btn-outline',!on);"
        ),
    )

## Sync JS Generator Wrapper

Convenience wrapper around `generate_card_stack_sync_js()` using the
segment-align constants. Returns a `Script` element ready for inclusion
in the page layout.

In [ ]:
#| export
def generate_sync_script(
    source_input_id:str,  # Seg card stack's focused_index hidden input ID
    target_nav_url:str,  # Align card stack's nav_to_index URL
) -> Any:  # Script element with sync JS
    """Generate the sync JS Script element for the combined step."""
    return Script(generate_card_stack_sync_js(
        source_input_id=source_input_id,
        target_nav_url=target_nav_url,
        toggle_fn_name=SYNC_TOGGLE_FN,
        sync_key=SYNC_KEY,
    ))

## KeyAction Toggle Wrapper JS

Named `window` function for the `S` key `KeyAction`. Toggles sync state
and updates the toolbar button styling.

In [ ]:
#| export
def generate_sync_key_toggle_js() -> str:  # JS defining window._segAlignSyncKeyToggle
    """Generate JS for the S key sync toggle wrapper function."""
    return f"""
    window._segAlignSyncKeyToggle = function() {{
        var on = window.{SYNC_TOGGLE_FN} && window.{SYNC_TOGGLE_FN}();
        var btn = document.getElementById('{SYNC_BTN_ID}');
        if (btn) {{
            btn.classList.toggle('btn-primary', on);
            btn.classList.toggle('btn-outline', !on);
        }}
    }};
    """

## Should-Play Guard Function

Named `window` function used by the web-audio settle handler to decide
whether to play audio. Replaces the default zone guard with a check that
allows playback when: zone is active, auto-navigate is on, or sync is enabled.

In [ ]:
#| export
# Function name passed to WebAudioConfig.should_play_fn via vad-align
SHOULD_PLAY_FN = "shouldAlignPlay"

def generate_should_play_js() -> str:  # JS defining window.shouldAlignPlay
    """Generate JS for the custom play guard function.
    
    Returns true when audio should play:
    - Zone is active (direct align navigation), OR
    - Auto-navigate is on (auto-play through align stack), OR
    - Sync is enabled (seg navigation driving align)
    """
    from cjm_transcript_vad_align.components.callbacks import ALIGN_AUDIO_CONFIG
    sk = ALIGN_AUDIO_CONFIG.state_key
    return f"""
    window.{SHOULD_PLAY_FN} = function(cardStackId) {{
        // Zone active check
        var zoneActive = true;
        if (window.kbNav) {{
            var kbState = window.kbNav.getState();
            if (kbState) zoneActive = (kbState.activeZoneId === cardStackId);
        }}
        if (zoneActive) return true;

        // Auto-navigate check
        if (window.{sk} && window.{sk}.autoNavigate) return true;

        // Sync enabled check
        if (window.{SYNC_KEY} && window.{SYNC_KEY}.enabled) return true;

        return false;
    }};"""

## Extra Actions Builder

Combines optional FA controls and sync toggle into a tuple for the
segmentation toolbar's `extra_actions` parameter.

## Sync Button Restore JS

Syncs the toolbar button styling with the client-side sync state after
a chrome switch re-renders the seg toolbar. Same pattern as the auto-play
toggle restore in `_restore_align_auto_nav_js()`.

In [ ]:
#| export
def generate_sync_restore_js() -> str:  # JS to restore sync button styling from client state
    """Generate JS to sync the toolbar button styling with the client-side sync state.
    
    After chrome switch re-renders the seg toolbar, the sync button starts
    with btn-outline. This reads the JS state and restores btn-primary if
    sync is enabled.
    """
    return f"""
        var _sb = document.getElementById('{SYNC_BTN_ID}');
        if (_sb && window.{SYNC_KEY}) {{
            var _on = window.{SYNC_KEY}.enabled || false;
            _sb.classList.toggle('btn-primary', _on);
            _sb.classList.toggle('btn-outline', !_on);
        }}
    """

In [ ]:
#| export
def build_extra_actions(
    fa_extra:Any=None,  # FA controls element (from build_fa_extra_actions, or None)
) -> tuple:  # Tuple of toolbar extra action elements
    """Build the extra_actions tuple for the seg toolbar.
    
    Combines FA controls (if available) with the sync toggle button.
    Client-side state restoration (sync button styling) is handled by
    the centralized toolbar restore settle handler in toolbar_state.py.
    Returns a tuple compatible with render_toolbar(extra_actions=...).
    """
    actions = []
    if fa_extra is not None:
        actions.append(fa_extra)
    actions.append(render_sync_toggle_button())
    return tuple(actions)

## Tests

In [ ]:
from fasthtml.common import Div, to_xml

# Test button renders with correct attributes
html = to_xml(render_sync_toggle_button())
assert SYNC_BTN_ID in html
assert 'type="button"' in html
assert SYNC_TOGGLE_FN in html
assert 'btn-outline' in html
assert 'Sync' in html

# Test sync script generation
script = generate_sync_script('seg-idx', '/align/nav')
script_html = to_xml(script)
assert 'seg-idx' in script_html
assert '/align/nav' in script_html
assert SYNC_TOGGLE_FN in script_html
assert SYNC_KEY in script_html

# Test key toggle JS
key_js = generate_sync_key_toggle_js()
assert '_segAlignSyncKeyToggle' in key_js
assert SYNC_TOGGLE_FN in key_js
assert SYNC_BTN_ID in key_js

# Test sync restore JS
restore_js = generate_sync_restore_js()
assert SYNC_BTN_ID in restore_js
assert SYNC_KEY in restore_js
assert 'btn-primary' in restore_js
assert 'btn-outline' in restore_js

# Test build_extra_actions — with FA (2 elements: FA + button)
fa_el = Div("fa", id="fa-slot")
actions = build_extra_actions(fa_extra=fa_el)
assert len(actions) == 2
assert SYNC_BTN_ID in to_xml(actions[1])

# Test build_extra_actions — without FA (1 element: button only)
actions_no_fa = build_extra_actions()
assert len(actions_no_fa) == 1
assert SYNC_BTN_ID in to_xml(actions_no_fa[0])

print('Sync controls tests passed')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()